# Simplifying Delegation in Python 

- toc: true
- badges: true
- comments: true
- author: Anish Dalal
- categories: [python,delegation]
- published: false

https://www.fast.ai/2019/08/06/delegation/

In [67]:
import inspect
from datetime import datetime

In [7]:
def delegates(to=None, keep=False):
    "Decorator: replace `**kwargs` in signature with params from `to`"
    def _f(f):
        if to is None: to_f,from_f = f.__base__.__init__,f.__init__
        else:          to_f,from_f = to,f
        sig = inspect.signature(from_f)
        sigd = dict(sig.parameters)
        k = sigd.pop('kwargs')
        s2 = {k:v for k,v in inspect.signature(to_f).parameters.items()
              if v.default != inspect.Parameter.empty and k not in sigd}
        sigd.update(s2)
        if keep: sigd['kwargs'] = k
        from_f.__signature__ = sig.replace(parameters=sigd.values())
        return f
    return _f

In [8]:
def custom_dir(c, add): return dir(type(c)) + list(c.__dict__.keys()) + add

class GetAttr:
    "Base class for attr accesses in `self._xtra` passed down to `self.default`"
    @property
    def _xtra(self): return [o for o in dir(self.default) if not o.startswith('_')]
    def __getattr__(self,k):
        if k in self._xtra: return getattr(self.default, k)
        raise AttributeError(k)
    def __dir__(self): return custom_dir(self, self._xtra)

In [64]:
def create_web_page(title, category="General", date=None, author="Jeremy"):
    pass

@delegates(create_web_page)
def create_product_page(title, price, cost, **kwargs):
    print(kwargs)

In [65]:
print(inspect.signature(create_web_page))
print(inspect.signature(create_product_page))

(title, category='General', date=None, author='Jeremy')
(title, price, cost, category='General', date=None, author='Jeremy')


In [68]:
class WebPage:
    def __init__(self, title, category="General", date=None, author="Jeremy"):
        self.title,self.category,self.author = title,category,author
        self.date = date or datetime.now()

@delegates()
class ProductPage(WebPage):
    def __init__(self, title, price, cost, **kwargs):
        super().__init__(title, **kwargs)

In [69]:
page = WebPage('Soap', category='Bathroom', author="Sylvain")
p = ProductPage(page, 15.0, 10.50)

In [73]:
p.date

datetime.datetime(2021, 2, 9, 19, 39, 45, 939863)

https://www.michaelcho.me/article/method-delegation-in-python